# 05 Net Conversion 非劣效决策

- **说明**：非劣效边界 δ 在分析前锁定（2026-08-31，见 config），本 Notebook 只引用、不做事后调整；另附"不同 δ 是否翻转结论"的敏感性分析，正式结论只用锁定值。
- **说明**：非劣效检验为**单侧 α=0.025**，等价于用 **95% 双侧 CI 的下界**做保守判定：**CI 下界 > −δ ⇒ 满足非劣效**。普通双侧显著性检验仍用 α=0.05，两套 α 口径**不混用**。
- **说明**：Gross×Net 非劣效决策矩阵，结合实际效应量解释。
- **说明**：非劣效核心图（点估计、95%CI、零线、−δ 边界、结论标注）。
- 同一口径：凡涉及"筛选减少注册/试听"，一律写"点估计与方向三法一致、显著性强度依赖 click 独立假设（day-cluster 口径下 Gross CI 跨 0）"。

In [1]:
# 加载锁定参数与前序主结果
import json, tomllib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
CFG = tomllib.load(open(ROOT/"config"/"analysis_config.toml","rb"))
LK = CFG["locked"]
DELTA = LK["ni_delta_net"]
NI_ALPHA = LK["ni_alpha_one_sided"]
main = json.loads((ROOT/"data"/"processed"/"main_effects.json").read_text(encoding="utf-8"))
net = main["z_tests"]["NetConversion"]; gross = main["z_tests"]["GrossConversion"]
d, lo, hi = net["abs_diff"], net["ci95_low"], net["ci95_high"]
se = net["se_ci_unpooled"]
print("锁定 δ =", DELTA, "| 非劣效单侧 α =", NI_ALPHA, "（等价 95% 双侧 CI 下界判定）")
print("Net: point=%.6f  95%%CI=(%.6f, %.6f)" % (d, lo, hi))

锁定 δ = 0.0075 | 非劣效单侧 α = 0.025 （等价 95% 双侧 CI 下界判定）
Net: point=-0.004874  95%CI=(-0.011604, 0.001857)


## 事前锁定的 δ（引用，不调整）+ 允许的 δ 敏感性

In [2]:
# 锁定 δ 的业务理由（引用 config 中锁定的值）
print(CFG["locked"]["ni_delta_rationale"])
# δ 敏感性（仅展示结论在何阈值翻转；正式结论只用 δ=0.75pp）
crit = -lo  # CI 下界 > -δ  <=>  δ > -lo
grid_d = [0.005, 0.0075, 0.010, crit, 0.0125, 0.015]
rows = []
for dd in grid_d:
    rows.append({"delta_pp": round(dd*100, 3),
                 "lower_bound > -delta": bool(lo > -dd),
                 "NI_verdict": "pass" if lo > -dd else "fail"})
sens = pd.DataFrame(rows)
print("\n临界 δ（CI 下界恰好等于 -δ）= %.3f pp；锁定 δ=0.75pp。" % (crit*100))
print(sens.to_string(index=False))

免费试听筛选器目标是过滤低投入意愿用户、降低运营成本；只要最终付费转化率下降不超过 0.75 个百分点，成本节约的价值就超过可接受的收入损失，因此 0.75pp 是本次实验的业务容忍边界。

临界 δ（CI 下界恰好等于 -δ）= 1.160 pp；锁定 δ=0.75pp。
 delta_pp  lower_bound > -delta NI_verdict
     0.50                 False       fail
     0.75                 False       fail
     1.00                 False       fail
     1.16                 False       fail
     1.25                  True       pass
     1.50                  True       pass


## 非劣效 α 口径与判定（单侧 0.025 ⇔ 95% 双侧 CI 下界）

In [3]:
# 两种等价表达：(1) CI 下界 > -δ；(2) 单侧 Z_NI=(d+δ)/SE 与 z_{1-0.025}=1.96
z_ni = (d + DELTA)/se
p_ni = stats.norm.sf(z_ni)               # 单侧上尾 p
zcrit = stats.norm.ppf(1-NI_ALPHA)
ni_pass = lo > -DELTA
print("判定式(1)：CI 下界 %.6f  >  -δ=%.6f ? %s" % (lo, -DELTA, lo > -DELTA))
print("判定式(2)：Z_NI=(d+δ)/SE=%.4f，临界 z=%.4f，单侧 p=%.4f -> %s"
      % (z_ni, zcrit, p_ni, "pass" if z_ni > zcrit else "fail"))
assert (z_ni > zcrit) == ni_pass  # 两式必须一致
print("点估计 d=%.3fpp 位于容忍区间内（|d|<δ），但 CI 下界 -1.160pp 越过 -0.75pp："
      "属于'精度不足以证明非劣效'，不是'证明了劣效'。" % (d*100,))

判定式(1)：CI 下界 -0.011604  >  -δ=-0.007500 ? False
判定式(2)：Z_NI=(d+δ)/SE=0.7648，临界 z=1.9600，单侧 p=0.2222 -> fail
点估计 d=-0.487pp 位于容忍区间内（|d|<δ），但 CI 下界 -1.160pp 越过 -0.75pp：属于'精度不足以证明非劣效'，不是'证明了劣效'。


## 非劣效决策矩阵（结合实际效应量）
Gross 列按**事前指定的 Z 主分析**判定为"显著下降"（同一口径：点估计与方向三法一致、显著性强度依赖 click 独立假设，day-cluster 口径下 Gross CI 跨 0）；Net 列按判定。

In [4]:
gross_sig = gross["p_value"] < LK["alpha_two_sided"] and gross["ci95_high"] < 0
matrix = pd.DataFrame([
    ["显著下降", "满足非劣效", "支持上线"],
    ["显著下降", "不满足非劣效", "拒绝上线"],
    ["未显著下降", "满足非劣效", "策略价值证据不足，考虑继续实验"],
    ["未显著下降", "不满足非劣效", "不建议上线"],
], columns=["Gross Conversion", "Net Conversion", "决策"])
g_label = "显著下降" if gross_sig else "未显著下降"
n_label = "满足非劣效" if ni_pass else "不满足非劣效"
hit = matrix[(matrix["Gross Conversion"]==g_label) & (matrix["Net Conversion"]==n_label)]
print(matrix.to_string(index=False))
print("\n本实验落入：Gross=%s，Net=%s -> 矩阵决策：%s" % (g_label, n_label, hit["决策"].iloc[0]))
print('''
结合效应量的解释（避免过度解读）：
1) Net 点估计 -0.49pp 本身在 0.75pp 容忍边界内，方向为负但幅度小；
2) 未通过非劣效门的原因是精度不足，
   CI 同时覆盖“损失小于边界”与“损失超过边界”，故为“未能确立非劣效”，不等于“已证劣效”；
3) 机械按矩阵=拒绝（全量）上线；与补样天数衔接，后续再综合成本收益给最终策略建议。''')

Gross Conversion Net Conversion              决策
            显著下降          满足非劣效            支持上线
            显著下降         不满足非劣效            拒绝上线
           未显著下降          满足非劣效 策略价值证据不足，考虑继续实验
           未显著下降         不满足非劣效           不建议上线

本实验落入：Gross=显著下降，Net=不满足非劣效 -> 矩阵决策：拒绝上线

结合效应量的解释（避免过度解读）：
1) Net 点估计 -0.49pp 本身在 0.75pp 容忍边界内，方向为负但幅度小；
2) 未通过非劣效门的原因是精度不足，
   CI 同时覆盖“损失小于边界”与“损失超过边界”，故为“未能确立非劣效”，不等于“已证劣效”；
3) 机械按矩阵=拒绝（全量）上线；与补样天数衔接，后续再综合成本收益给最终策略建议。


In [5]:
# 非劣效核心图
fig, ax = plt.subplots(figsize=(9.2, 3.6), dpi=150)
# 非劣效可接受区域：效应 > -δ
ax.axvspan(-DELTA*100, 1.2, color="#2ca02c", alpha=.10)
ax.axvline(0, color="black", lw=1.3)
ax.axvline(-DELTA*100, color="#d62728", ls="--", lw=1.6)
ax.errorbar(d*100, 0, xerr=[[(d-lo)*100], [(hi-d)*100]], fmt="o", color="#1f4e79",
            capsize=6, lw=2.4, markersize=9)
ax.text(d*100, 0.22, f"point={d*100:+.2f}pp", ha="center", fontsize=9)
ax.text(lo*100, -0.30, f"lower={lo*100:+.2f}pp", ha="center", fontsize=9, color="#1f4e79")
ax.text(-DELTA*100, 0.42, "-delta = -0.75pp (NI boundary)", color="#d62728", fontsize=9, ha="center")
ax.text(0, 0.42, "zero effect", color="black", fontsize=9, ha="center")
verdict = ("NI NOT established: lower bound -1.16pp < -0.75pp (one-sided alpha=0.025)\n"
           "inconclusive due to limited precision - NOT proof of inferiority")
ax.text(0.5, -0.72, verdict, fontsize=9.5, ha="center",
        bbox=dict(boxstyle="round", fc="#fff3cd", ec="#d62728"))
ax.set_ylim(-1, 0.8); ax.set_xlim(-2.2, 1.2); ax.set_yticks([])
ax.set_xlabel("Net Conversion absolute difference (Experiment - Control), pp")
ax.set_title("Non-inferiority assessment of Net Conversion (locked delta=0.75pp)")
ax.grid(axis="x", alpha=.3)
fig.tight_layout(); fig.savefig(ROOT/"reports"/"figures"/"fig_noninferiority.png"); plt.close(fig)
print("saved reports/figures/fig_noninferiority.png")

saved reports/figures/fig_noninferiority.png


In [6]:
# 落盘 + 回读
out = {"locked_delta": DELTA, "ni_alpha_one_sided": NI_ALPHA,
       "rule": "NI pass iff 95% two-sided CI lower bound > -delta (equivalent to one-sided alpha=0.025)",
       "net_point": float(d), "net_ci95": [float(lo), float(hi)],
       "z_ni": float(z_ni), "p_ni_one_sided": float(p_ni),
       "ni_pass": bool(ni_pass), "critical_delta_to_pass": float(crit),
       "gross_verdict_primary_ztest": g_label, "matrix_decision": hit["决策"].iloc[0],
       "delta_sensitivity": rows}
p = ROOT/"data"/"processed"/"ni_decision.json"
p.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
back = json.loads(p.read_text(encoding="utf-8"))
assert back["ni_pass"] == ni_pass
print("ni_decision.json written & re-read OK; ni_pass =", back["ni_pass"])

ni_decision.json written & re-read OK; ni_pass = False


## 小结
1. **δ=0.75pp 为事前锁定**，业务理由逐字存档；敏感性显示 CI 下界对应的临界 δ≈1.16pp，正式结论只采用 0.75pp。
2. **非劣效未通过**：Net 95% CI 下界 −1.160pp < −0.75pp（等价 Z_NI=0.77 < 1.96，单侧 p≈0.22）。点估 −0.49pp 在容忍区内，但实验 underpowered、精度不足，结论是**"未能确立非劣效"而非"已证劣效"**。
3. 决策矩阵机械落入"Gross 显著下降（Z 主分析，显著性强度依赖 click 独立假设）+ Net 不满足非劣效 → 拒绝（全量）上线"；最终策略建议留待后续分析 结合成本收益与补样方案综合给出。